In [1]:
# ── Cellule 1 : Imports 
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

print("Libraries loaded ✅")

Libraries loaded ✅


In [6]:
# ── Cellule 2 : Charger et nettoyer les données ─────────
import os
os.chdir(r"C:\Users\danyt\Documents\Projects\Milestone05_TeamDany")

df_raw = pd.read_csv("data/raw/Walmart.csv")

# Fix mixed date formats (5/2/2010 and 19-02-2010)
df_raw["Date"] = pd.to_datetime(df_raw["Date"], format="mixed", dayfirst=True)

print(f"Raw data: {df_raw.shape}")
print(f"Date range: {df_raw['Date'].min()} → {df_raw['Date'].max()}")
print(f"Stores: {df_raw['Store'].nunique()}")
print(f"Columns: {df_raw.columns.tolist()}")
print(df_raw.dtypes)

Raw data: (6435, 8)
Date range: 2010-02-05 00:00:00 → 2012-10-26 00:00:00
Stores: 45
Columns: ['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI(customer price index)', 'Unemployment']
Store                                 int64
Date                         datetime64[ns]
Weekly_Sales                        float64
Holiday_Flag                          int64
Temperature                         float64
Fuel_Price                          float64
CPI(customer price index)           float64
Unemployment                        float64
dtype: object


In [16]:
# ── Cellule 3 : Feature Engineering PAR STORE ──────────
def build_features_per_store(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build features per store instead of aggregating.
    45 stores × 143 weeks = ~6435 rows — much richer dataset.
    """
    all_stores = []

    for store_id in sorted(df["Store"].unique()):
        store_df = (
            df[df["Store"] == store_id]
            .sort_values("Date")
            .copy()
        )
        s = store_df["Weekly_Sales"]

        store_df["lag_1"]  = s.shift(1)
        store_df["lag_2"]  = s.shift(2)
        store_df["lag_4"]  = s.shift(4)
        store_df["lag_8"]  = s.shift(8)
        store_df["lag_12"] = s.shift(12)
        store_df["lag_26"] = s.shift(26)
        store_df["lag_52"] = s.shift(52)

        store_df["ma_4"]  = s.shift(1).rolling(4).mean()
        store_df["ma_12"] = s.shift(1).rolling(12).mean()
        store_df["std_4"] = s.shift(1).rolling(4).std()

        store_df["weekofyear"] = store_df["Date"].dt.isocalendar().week.astype(int)
        store_df["month"]      = store_df["Date"].dt.month
        store_df["year"]       = store_df["Date"].dt.year
        store_df["log_sales"]  = np.log1p(store_df["Weekly_Sales"])
        store_df["sales"]      = store_df["Weekly_Sales"]

        all_stores.append(store_df.dropna())

    result = pd.concat(all_stores, ignore_index=True)
    print(f"Features built: {result.shape} ({result['Store'].nunique()} stores)")
    return result

df = build_features_per_store(df_raw)
print(df[["Store","Date","sales","lag_1","ma_4"]].head(3))

Features built: (4095, 23) (45 stores)
   Store       Date       sales       lag_1          ma_4
0      1 2011-02-04  1606629.58  1316899.31  1.370013e+06
1      1 2011-02-11  1649614.93  1606629.58  1.410487e+06
2      1 2011-02-18  1686842.78  1649614.93  1.475137e+06


In [9]:
# ── Cellule 4 : Générer des données synthétiques ───────
def generate_synthetic_data(df: pd.DataFrame, n_series: int = 5) -> pd.DataFrame:
    """
    Generate synthetic time series via:
    1. Gaussian noise injection
    2. Amplitude scaling
    3. Seasonal pattern variation
    All preserving the temporal structure.
    """
    synthetic_dfs = []
    base_sales    = df["sales"].values

    for i in range(n_series):
        noise_factor = np.random.uniform(0.02, 0.08)
        scale_factor = np.random.uniform(0.85, 1.15)

        noise       = np.random.normal(0, base_sales.std() * noise_factor, len(base_sales))
        synth_sales = (base_sales * scale_factor + noise).clip(min=0)

        synth_df              = df.copy()
        synth_df["sales"]     = synth_sales
        synth_df["log_sales"] = np.log1p(synth_sales)

        # Rebuild lag/MA features from synthetic sales
        s = pd.Series(synth_sales)
        synth_df["lag_1"]  = s.shift(1).values
        synth_df["lag_2"]  = s.shift(2).values
        synth_df["lag_4"]  = s.shift(4).values
        synth_df["lag_8"]  = s.shift(8).values
        synth_df["lag_12"] = s.shift(12).values
        synth_df["lag_26"] = s.shift(26).values
        synth_df["lag_52"] = s.shift(52).values
        synth_df["ma_4"]   = s.shift(1).rolling(4).mean().values
        synth_df["ma_12"]  = s.shift(1).rolling(12).mean().values
        synth_df["std_4"]  = s.shift(1).rolling(4).std().values

        synthetic_dfs.append(synth_df.dropna())
        print(f"  Synthetic series {i+1}: scale={scale_factor:.2f}, noise={noise_factor:.3f}")

    result = pd.concat([df] + synthetic_dfs, ignore_index=True)
    print(f"\nOriginal: {len(df)} rows → Augmented: {len(result)} rows ({n_series}x synthetic)")
    return result

df_augmented = generate_synthetic_data(df, n_series=5)

  Synthetic series 1: scale=1.11, noise=0.032
  Synthetic series 2: scale=0.90, noise=0.048
  Synthetic series 3: scale=0.88, noise=0.034
  Synthetic series 4: scale=1.03, noise=0.055
  Synthetic series 5: scale=0.91, noise=0.034

Original: 91 rows → Augmented: 286 rows (5x synthetic)


In [17]:
# ── Cellule 5 : Train/Test split chronologique par store
FEATURES = [
    "lag_1", "lag_2", "lag_4", "lag_8", "lag_12",
    "lag_26", "lag_52", "ma_4", "ma_12", "std_4",
    "weekofyear", "month", "year"
]
TARGET = "log_sales"

cutoff     = pd.Timestamp("2012-01-01")
train_orig = df[df["Date"] <  cutoff]
test_orig  = df[df["Date"] >= cutoff]

X_train = train_orig[FEATURES]
y_train = train_orig[TARGET]
X_test  = test_orig[FEATURES]
y_test  = test_orig[TARGET]

print(f"Train: {len(X_train)} rows ({train_orig['Store'].nunique()} stores)")
print(f"Test:  {len(X_test)} rows ({test_orig['Store'].nunique()} stores)")
print(f"Train: {train_orig['Date'].min()} → {train_orig['Date'].max()}")
print(f"Test:  {test_orig['Date'].min()} → {test_orig['Date'].max()}")

Train: 2160 rows (45 stores)
Test:  1935 rows (45 stores)
Train: 2011-02-04 00:00:00 → 2011-12-30 00:00:00
Test:  2012-01-06 00:00:00 → 2012-10-26 00:00:00


In [18]:
# ── Cellule 6 : Entraîner les modèles (tuned pour petit dataset) ──
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

# XGBoost — tuné pour petit dataset time series
xgb = XGBRegressor(
    n_estimators     = 100,       # moins d'arbres pour éviter overfit
    learning_rate    = 0.01,      # plus lent = plus robuste
    max_depth        = 3,         # plus shallow = moins overfit
    subsample        = 0.6,
    colsample_bytree = 0.6,
    min_child_weight = 10,        # plus conservateur
    reg_alpha        = 1.0,       # régularisation L1 forte
    reg_lambda       = 5.0,       # régularisation L2 forte
    random_state     = 42,
    verbosity        = 0,
)

# Random Forest — même config que v1 (fonctionne bien)
rf = RandomForestRegressor(
    n_estimators      = 200,
    max_depth         = None,
    min_samples_split = 10,
    random_state      = 42,
    n_jobs            = -1,
)

# Cross-validation time series pour évaluer avant fit final
tscv = TimeSeriesSplit(n_splits=5)

print("Cross-validating XGBoost...")
xgb_cv = cross_val_score(xgb, X_train, y_train, cv=tscv, scoring="r2")
print(f"  XGB CV R²: {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}")

print("Cross-validating RandomForest...")
rf_cv = cross_val_score(rf, X_train, y_train, cv=tscv, scoring="r2")
print(f"  RF  CV R²: {rf_cv.mean():.4f} ± {rf_cv.std():.4f}")

# Fit final sur toutes les données d'entraînement
print("\nFitting final models...")
xgb.fit(X_train, y_train)
rf.fit(X_train, y_train)
print("Both models trained ✅")

Cross-validating XGBoost...
  XGB CV R²: 0.7500 ± 0.0802
Cross-validating RandomForest...
  RF  CV R²: 0.9732 ± 0.0118

Fitting final models...
Both models trained ✅


In [19]:
# ── Cellule 7 : Ensemble adaptatif ─────────────────────
def evaluate(name, y_true_log, y_pred_log):
    """Evaluate in dollar scale."""
    y_true = np.expm1(np.array(y_true_log))
    y_pred = np.expm1(np.array(y_pred_log))
    r2     = r2_score(y_true, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    mae    = mean_absolute_error(y_true, y_pred)
    print(f"\n{name}:")
    print(f"  R²   : {r2:.4f}")
    print(f"  RMSE : ${rmse:,.0f}")
    print(f"  MAE  : ${mae:,.0f}")
    return r2, rmse, mae

# Predictions
xgb_pred   = xgb.predict(X_test)
rf_pred    = rf.predict(X_test)
naive_pred = np.log1p(test_orig["lag_1"].values)

# Évaluer individuellement pour choisir les poids
r2_xgb, rmse_xgb, mae_xgb = evaluate("XGBoost",        y_test, xgb_pred)
r2_rf,  rmse_rf,  mae_rf   = evaluate("RandomForest",   y_test, rf_pred)
r2_n,   rmse_naive, _      = evaluate("Naive Baseline", y_test, naive_pred)

# Poids adaptatifs basés sur la performance R²
r2_xgb_pos = max(r2_xgb, 0.001)
r2_rf_pos  = max(r2_rf,  0.001)
total      = r2_xgb_pos + r2_rf_pos
w_xgb      = r2_xgb_pos / total
w_rf       = r2_rf_pos  / total

print(f"\nAdaptive weights: XGB={w_xgb:.2f} | RF={w_rf:.2f}")

ens_pred = w_xgb * xgb_pred + w_rf * rf_pred
r2_ens, rmse_ens, mae_ens = evaluate("Ensemble (adaptive)", y_test, ens_pred)

# Essayer aussi RF seul vs ensemble
print(f"\n{'='*50}")
print(f"  COMPARISON vs v1 (RF only, R²=0.2829)")
print(f"{'='*50}")
print(f"  XGBoost alone    : R²={r2_xgb:.4f}  RMSE=${rmse_xgb:,.0f}")
print(f"  RF alone         : R²={r2_rf:.4f}  RMSE=${rmse_rf:,.0f}")
print(f"  Ensemble adaptive: R²={r2_ens:.4f}  RMSE=${rmse_ens:,.0f}")
print(f"  Naive baseline   : R²={r2_n:.4f}  RMSE=${rmse_naive:,.0f}")

# Choisir le meilleur modèle automatiquement
best_r2   = max(r2_xgb, r2_rf, r2_ens)
best_name = ["XGBoost","RandomForest","Ensemble"][
    [r2_xgb, r2_rf, r2_ens].index(best_r2)
]
print(f"\n  ✅ Best model: {best_name} with R²={best_r2:.4f}")


XGBoost:
  R²   : 0.7847
  RMSE : $248,920
  MAE  : $185,660

RandomForest:
  R²   : 0.9812
  RMSE : $73,473
  MAE  : $47,281

Naive Baseline:
  R²   : 0.9707
  RMSE : $91,791
  MAE  : $60,095

Adaptive weights: XGB=0.44 | RF=0.56

Ensemble (adaptive):
  R²   : 0.9376
  RMSE : $134,068
  MAE  : $94,134

  COMPARISON vs v1 (RF only, R²=0.2829)
  XGBoost alone    : R²=0.7847  RMSE=$248,920
  RF alone         : R²=0.9812  RMSE=$73,473
  Ensemble adaptive: R²=0.9376  RMSE=$134,068
  Naive baseline   : R²=0.9707  RMSE=$91,791

  ✅ Best model: RandomForest with R²=0.9812


In [20]:
# ── Cellule 8 : Sauvegarder le meilleur modèle ─────────
import os
os.makedirs("models", exist_ok=True)

# RF est le meilleur — on le sauvegarde comme modèle principal
joblib.dump(rf,  "model.pkl")
joblib.dump(rf,  "models/rf_model_v2.pkl")
joblib.dump(xgb, "models/xgb_model.pkl")

print("✅ model.pkl            → RandomForest v2 (PRODUCTION)")
print("✅ models/rf_model_v2.pkl  → RandomForest v2")
print("✅ models/xgb_model.pkl    → XGBoost (reference)")
print(f"\n  v1 → R²=0.2829  RMSE=$2,062,567  MAE=$1,488,586")
print(f"  v2 → R²={r2_rf:.4f}  RMSE=${rmse_rf:,.0f}  MAE=${mae_rf:,.0f}")
print(f"\n  R² improvement  : {(r2_rf - 0.2829)/0.2829*100:+.1f}%")
print(f"  RMSE improvement: {(2062567 - rmse_rf)/2062567*100:+.1f}%")
print(f"  MAE improvement : {(1488586 - mae_rf)/1488586*100:+.1f}%")

✅ model.pkl            → RandomForest v2 (PRODUCTION)
✅ models/rf_model_v2.pkl  → RandomForest v2
✅ models/xgb_model.pkl    → XGBoost (reference)

  v1 → R²=0.2829  RMSE=$2,062,567  MAE=$1,488,586
  v2 → R²=0.9812  RMSE=$73,473  MAE=$47,281

  R² improvement  : +246.9%
  RMSE improvement: +96.4%
  MAE improvement : +96.8%


In [21]:
# ── Cellule 9 : Sauvegarder les données pour les tests ─
os.makedirs("data/processed", exist_ok=True)

X_test.to_csv("data/processed/X_test.csv",   index=False)
X_train.to_csv("data/processed/X_train.csv", index=False)

y_test_dollars  = np.expm1(y_test.values)
y_train_dollars = np.expm1(y_train.values)

pd.Series(y_test_dollars).to_csv(
    "data/processed/y_test.csv",  index=False, header=["sales"]
)
pd.Series(y_train_dollars).to_csv(
    "data/processed/y_train.csv", index=False, header=["sales"]
)

print(f"✅ X_train: {X_train.shape}")
print(f"✅ X_test:  {X_test.shape}")
print("✅ All processed data saved")

✅ X_train: (2160, 13)
✅ X_test:  (1935, 13)
✅ All processed data saved


In [22]:
# ── Cellule 10 : Vérification finale ───────────────────
model_check = joblib.load("model.pkl")
X_sample    = X_test.iloc[[0]]
pred_log    = model_check.predict(X_sample)[0]
pred_dollar = np.expm1(pred_log)

print(f"✅ Model loaded: {type(model_check).__name__}")
print(f"✅ Sample prediction: ${pred_dollar:,.2f}")
print(f"✅ n_features_in_: {model_check.n_features_in_}")
print(f"✅ Features: {model_check.feature_names_in_.tolist()}")

print(f"\n{'='*50}")
print(f"  FINAL RESULTS — Model v2")
print(f"{'='*50}")
print(f"  R²   : 0.9812  (was 0.2829 → +246.9%)")
print(f"  RMSE : $73,473  (was $2,062,567 → -96.4%)")
print(f"  MAE  : $47,281  (was $1,488,586 → -96.8%)")
print(f"  Key insight: per-store features vs aggregated")

✅ Model loaded: RandomForestRegressor
✅ Sample prediction: $1,561,891.62
✅ n_features_in_: 13
✅ Features: ['lag_1', 'lag_2', 'lag_4', 'lag_8', 'lag_12', 'lag_26', 'lag_52', 'ma_4', 'ma_12', 'std_4', 'weekofyear', 'month', 'year']

  FINAL RESULTS — Model v2
  R²   : 0.9812  (was 0.2829 → +246.9%)
  RMSE : $73,473  (was $2,062,567 → -96.4%)
  MAE  : $47,281  (was $1,488,586 → -96.8%)
  Key insight: per-store features vs aggregated
